# NREL WTK – Kahuku Wind Data 2017–2019
Run each cell in order from top to bottom.

In [ ]:
# install library 
import subprocess
subprocess.run(['pip', 'install', 's3fs', 'h5py', '-q'], check=True)
print('Done install')

In [ ]:
# Import
import s3fs
import h5py
import pandas as pd
import numpy as np
print('Import OK')

In [ ]:
# parameter
TARGET_LAT =  21.667046
TARGET_LON = -157.952130
YEARS = [2017, 2018, 2019]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = (np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

print('Target:', TARGET_LAT, TARGET_LON)

In [ ]:
# connect S3
fs = s3fs.S3FileSystem(anon=True)

def get_path(year):
    for p in [
        f'nrel-pds-wtk/Hawaii/Hawaii_{year}.h5',
        f'nrel-pds-wtk/v1.0.0/Hawaii/Hawaii_{year}.h5',
    ]:
        if fs.exists(p):
            return p
    raise FileNotFoundError(f'No H5 for {year}')

for y in YEARS:
    p = get_path(y)
    print(y, '->', p)

In [ ]:
# Download va resample hourly
all_years = []

for year in YEARS:
    path = get_path(year)
    print(f'\n=== {year} ===')

    with fs.open(path, 'rb') as f:
        with h5py.File(f, 'r') as h5:

            if year == 2017:
                print('Variables:', list(h5.keys()))

            coords = h5['coordinates'][:]
            lats, lons = coords[:,0], coords[:,1]
            idx = int(np.argmin(haversine(TARGET_LAT, TARGET_LON, lats, lons)))
            print(f'Grid idx={idx}  lat={lats[idx]:.4f}  lon={lons[idx]:.4f}')

            time_raw = h5['time_index'][:]
            if isinstance(time_raw[0], (bytes, np.bytes_)):
                time_raw = [t.decode() for t in time_raw]
            timestamps = pd.to_datetime(time_raw)

            # # NREL WTK stores wind speed as integer * 100; divide by 100 to recover m/s
            ws = h5['windspeed_100m'][:,    idx].astype(float) / 100.0
            wd = h5['winddirection_100m'][:, idx].astype(float) / 100.0

            pk = next((k for k in h5 if 'pressure'    in k.lower()), None)
            tk = next((k for k in h5 if 'temperature' in k.lower()), None)
            pres = h5[pk][:, idx].astype(float) if pk else np.full(len(ws), np.nan)
            temp = h5[tk][:, idx].astype(float) if tk else np.full(len(ws), np.nan)

            df5 = pd.DataFrame({
                'Time':                   timestamps,
                'WindSpeed_100m_mps':     ws,
                'WindDirection_100m_deg': wd,
                'Pressure_Pa':            pres,
                'Temperature':            temp,
                'Latitude':               lats[idx],
                'Longitude':              lons[idx],
            }).set_index('Time')

            dfh = df5.resample('1h').mean().reset_index()
            dfh = dfh[dfh['Time'].dt.year == year].head(8760).copy()

            dfh['Hour']      = dfh['Time'].dt.hour
            dfh['Day']       = dfh['Time'].dt.day
            dfh['Month']     = dfh['Time'].dt.month
            dfh['DayOfYear'] = dfh['Time'].dt.dayofyear
            dfh['WindSpeed_100m_mps'] = dfh['WindSpeed_100m_mps'].round(2)

            v = dfh['WindSpeed_100m_mps']
            print(f'rows={len(dfh)}  mean={v.mean():.2f}  std={v.std():.2f}  min={v.min():.2f}  max={v.max():.2f}')
            all_years.append(dfh)

print('\nAll years OK')

In [ ]:
#  save CSV
COLS = ['Time','WindSpeed_100m_mps','WindDirection_100m_deg',
        'Pressure_Pa','Latitude','Longitude','Temperature',
        'Hour','Day','Month','DayOfYear']

df_train    = all_years[0]
df_validate = all_years[1]
df_test     = all_years[2]
df_all      = pd.concat(all_years, ignore_index=True)

df_train.to_csv   ('train_wind_100m.csv',              columns=COLS, index=False)
df_validate.to_csv('validate_wind_100m.csv',           columns=COLS, index=False)
df_test.to_csv    ('test_wind_100m.csv',               columns=COLS, index=False)
df_all.to_csv     ('Hawaii_Kahuku_Wind_2017_2019.csv', columns=COLS, index=False)

print('Saved:')
for name, df in [('train',df_train),('validate',df_validate),
                 ('test',df_test),('all',df_all)]:
    print(f'  {name}: {len(df)} rows')

In [ ]:
# Quick sanity check before downloading
# Print summary statistics for all three splits

for name, df in [('Train 2017', df_train), 
                 ('Validate 2018', df_validate), 
                 ('Test 2019', df_test)]:
    v = df['WindSpeed_100m_mps']
    print(f'=== {name} ===')
    print(f'  rows  = {len(df)}')
    print(f'  mean  = {v.mean():.2f} m/s')
    print(f'  std   = {v.std():.2f} m/s')
    print(f'  min   = {v.min():.2f} m/s')
    print(f'  max   = {v.max():.2f} m/s')
    print(f'  nulls = {v.isnull().sum()}')
    print()

In [ ]:
# Download
from google.colab import files
for fname in ['train_wind_100m.csv',
              'validate_wind_100m.csv',
              'test_wind_100m.csv',
              'Hawaii_Kahuku_Wind_2017_2019.csv']:
    files.download(fname)
    print('Downloading', fname)